In [2]:
import cv2
import numpy as np
from insightface.app import FaceAnalysis

# Initialize face analysis model
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])  # Use 'CUDAExecutionProvider' for GPU
app.prepare(ctx_id=-0)  # ctx_id=-1 for CPU, 0 for GPU

def get_face_embedding(image_path):
    """Extract face embedding from an image"""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")
    
    faces = app.get(img)
    
    if len(faces) < 1:
        raise ValueError("No faces detected in the image")
    if len(faces) > 1:
        print("Warning: Multiple faces detected. Using first detected face")
    
    return faces[0].embedding

c:\Users\Sam\Desktop\project_ia_face\faceenv\lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.7' (you have '2.0.6'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\Sam/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'],

## Propuesta 2: Comparación de embedding con múltiples embeddings
-> Mayor coste computacional, mayor precisión

Haremos uso de la librería Faiss aunque realmente para este contexto bastaría con numpy
Para el escenario de comparación con múltiples embeddings Faiss es muy importante.
Consideremos que cada usuario tendrá asociado 4 embeddings con variaciones de luminosidad, artefactos o expresiones faciales,
la comparación pasaría a ser: 1 -> n * 4 embeddings, siendo n el número de usuarios y 4 la cantidad de embeddings de por usuarios
Faiss indexaría la base de datos completa con embeddings normalizados, y se compararía el embedding de consulta con cada uno de estos.
El usuario en caso de estar registrado tendría un embedding que coincidiría con cualquiera de los 4 asociados a este.

In [3]:
#Faiss -> Más aplicable al uso de miles de comparaciones con múltiples embeddings

import faiss
import numpy as np


#Embeddings que simulan pertenecer a la base de datos
emb1 = get_face_embedding("../img_test/compare/ana2.jpg")
emb2 = get_face_embedding("../img_test/compare/ana3.jpg")
emb3 = get_face_embedding("../img_test/compare/ana4.jpg")
emb4 = get_face_embedding("../img_test/compare/ana5.jpg")

#Embedding de consulta
emb_query = get_face_embedding("../img_test/compare/ana1.jpg")


# Base de embeddings del individuo
known_embs = np.array([emb1, emb2, emb3, emb4], dtype='float32')
known_embs /= np.linalg.norm(known_embs, axis=1, keepdims=True)  # Normalizar para coseno

"""
    axis=1 indica que estás normalizando cada fila individualmente (cada vector).
    keepdims=True mantiene la forma (N, 1) para que se pueda hacer la división por broadcasting correctamente.
"""

# Embedding a comparar
query = emb_query.astype('float32')
query /= np.linalg.norm(query)

# Crear índice FAISS
index = faiss.IndexFlatIP(512)  # Inner product (cosine similarity si está normalizado) (512 es la dimensionalidad empleada por insightface)
"""
El producto interno entre dos vectores es exactamente igual a la similitud por coseno de estas:
"""
index.add(known_embs)

# Buscar el más parecido
similarities, indices = index.search(np.expand_dims(query, axis=0), k=1)

print("Mejor score:", similarities[0][0])
print("Embedding con mejor score: ", indices[0][0])



Mejor score: 0.7548983
Embedding con mejor score:  1
